# Playing with `piepy.stats.aggregate`

The aggregator turns a trial table into a **tidy** per-condition estimate frame — one row
per `(group × metric)` with `value`, `ci_low`, `ci_high`, `n`. It is experiment-agnostic
(you name the grouping columns and the metrics) and the output is plotting-ready.

This notebook runs on **synthetic data** so it works anywhere; the final cell shows an
optional real-session example.

In [ ]:
import numpy as np
import polars as pl
import matplotlib.pyplot as plt

from piepy.stats import aggregate, group_arrays, Rate, Mean, Median, Count

pl.Config(set_tbl_rows=20)

## A synthetic detection dataset

3 animals × a range of signed contrasts. Hit rate rises with |contrast| (detection-style),
reaction times are gamma-distributed and a little slower on opto trials. Tweak anything here
and re-run the cells below.

In [ ]:
rng = np.random.default_rng(0)
contrasts = [-1.0, -0.5, -0.25, -0.125, 0.0, 0.125, 0.25, 0.5, 1.0]
animals = ["M1", "M2", "M3"]

rows = []
for animal in animals:
    slope = rng.uniform(7, 12)
    lapse = rng.uniform(0.02, 0.08)
    for c in contrasts:
        n = int(rng.integers(25, 45))
        p_hit = lapse + (1 - 2 * lapse) / (1 + np.exp(-slope * (abs(c) - 0.15)))
        for _ in range(n):
            hit = rng.random() < p_hit
            opto = rng.random() < 0.3
            rt = 150 + rng.gamma(2.0, 120) + (80 if opto else 0)
            rows.append(
                dict(
                    animalid=animal,
                    signed_contrast=c,
                    is_hit=int(hit),
                    outcome="hit" if hit else "miss",
                    opto=opto,
                    reaction_time=float(rt),
                )
            )

df = pl.DataFrame(rows)
print(df.shape)
df.head()

## 1. Hit rate per condition (shorthand)

`rate="is_hit"` → proportion with a **Wilson** CI, which stays inside `[0, 1]` even at
0% / 100%. The result is tidy: `signed_contrast | metric | value | ci_low | ci_high | n`.

In [ ]:
psych = aggregate(df, group="signed_contrast", rate="is_hit").sort("signed_contrast")
psych

## 2. The output is plotting-ready

`value` plus the `ci_low`/`ci_high` columns drop straight into an errorbar. In the real
pipeline behaviz draws this — here we use matplotlib so the notebook is self-contained.

In [ ]:
x = psych["signed_contrast"].to_numpy()
y = psych["value"].to_numpy()
yerr = np.vstack([y - psych["ci_low"].to_numpy(), psych["ci_high"].to_numpy() - y])

plt.figure(figsize=(6, 4))
plt.errorbar(x, y, yerr=yerr, marker="o", capsize=3)
plt.ylim(0, 1)
plt.xlabel("signed contrast")
plt.ylabel("hit rate")
plt.title("Psychometric (synthetic) — Wilson 95% CI")
plt.show()

## 3. Median reaction time per condition

`value="reaction_time", stat="median"` → median with the analytic **order-statistic** CI
(fast, deterministic, no bootstrap loop). Prefer bootstrap? use
`metrics=[Median("reaction_time", ci="bootstrap")]`.

In [ ]:
aggregate(df, group="signed_contrast", value="reaction_time", stat="median").sort(
    "signed_contrast"
)

## 4. Several metrics in one pass

Pass a list of metric specs; the output stays tidy (one row per group × metric).

In [ ]:
aggregate(
    df,
    group=["animalid","opto"],
    metrics=[Rate("is_hit"), Median("reaction_time"), Mean("reaction_time"), Count()],
).sort(["animalid","opto", "metric"])

## 5. Group by anything — e.g. animal × condition

Just add columns to `group`. Here: one psychometric per animal.

In [ ]:
per = aggregate(df, group=["animalid", "signed_contrast"], rate="is_hit").sort(
    ["animalid", "signed_contrast"]
)

plt.figure(figsize=(6, 4))
for animal in per["animalid"].unique(maintain_order=True):
    sub = per.filter(pl.col("animalid") == animal)
    xx, yy = sub["signed_contrast"].to_numpy(), sub["value"].to_numpy()
    err = np.vstack([yy - sub["ci_low"].to_numpy(), sub["ci_high"].to_numpy() - yy])
    plt.errorbar(xx, yy, yerr=err, marker="o", capsize=2, label=animal)
plt.ylim(0, 1)
plt.xlabel("signed contrast")
plt.ylabel("hit rate")
plt.legend(title="animal")
plt.title("Per-animal psychometrics")
plt.show()

## 6. Rate from a categorical column

`Rate("outcome", success="hit")` works off a string column — same result as the 0/1 `is_hit`.

In [ ]:
aggregate(df, group="signed_contrast", metrics=[Rate("outcome", success="hit")]).sort(
    "signed_contrast"
)

## 7. Raw arrays for statistical tests

`group_arrays` hands you the per-group arrays to feed a test. (Chunk 2 will add
`piepy.stats.compare`; here we preview with scipy directly.)

In [ ]:
from scipy.stats import mannwhitneyu

arrs = group_arrays(df, group="opto", value="reaction_time")
u, p = mannwhitneyu(arrs[True], arrs[False])
print(f"opto (n={arrs[True].size}) vs control (n={arrs[False].size}) reaction time")
print(f"Mann-Whitney U={u:.0f}, p={p:.3g}")

## 8. Robustness

Empty groups become `n=0` with `null` estimates (never a crash), null group **keys** are
kept as their own group, and a missing column raises a clear error.

In [ ]:
demo = pl.DataFrame({"cond": ["a", "a", "b", None], "rt": [1.0, 2.0, None, 5.0]})
aggregate(demo, group="cond", value="rt", stat="median")  # b has no rt -> n=0; null key kept

## (Optional) A real session

Needs lab data + `~/.piepy/config.json`. Analysis writes are redirected to a temp dir so
nothing is clobbered, and the cell skips cleanly if the session isn't available locally.

In [ ]:
import tempfile
from piepy.core.config import config

config.paths["analysis"] = [tempfile.mkdtemp()]  # do not touch real analysis output
config.verbose = False  # quiet the parse logs / progress bars

try:
    from piepy.psychophysics.wheel.detection.wheelDetectionSession import (
        WheelDetectionSession,
    )

    sess = WheelDetectionSession("230106_KC144_detect__no_cam_KC", load_flag=False)
    real = sess.concatenate_runs("wheel_detection")
    out = aggregate(
        real,
        group="signed_contrast",
        metrics=[Rate("outcome", success="hit"), Median("reaction_time"), Count()],
    ).sort(["signed_contrast", "metric"])
    print(real.shape)
    display(out)
except Exception as e:
    print("Skipping real-data example:", type(e).__name__, e)